### SET UP

In [ ]:
import os
import sys
import gc
import json
import time
from datetime import datetime
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np

# Nạp các module tự viết từ thư mục lõi
sys.path.append(os.path.abspath("../"))
from core.module05_fusion_classifier.dataset import VariantFusionDataset
from core.module05_fusion_classifier.fusion_model import MultiStrategyFusionModel
from core.module05_fusion_classifier.xgboost_model import XGBoostFusionManager
from core.module05_fusion_classifier.evaluator_profiler import FusionEvaluatorProfiler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Đang sử dụng thiết bị: {DEVICE}")

# ==============================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN & MA TRẬN THÍ NGHIỆM
# ==============================================================================
REPO_ROOT = os.path.abspath("../")

# Trỏ Data Paths vào trong thư mục /data
BIO_DIR = f"{REPO_ROOT}/data/processed"          
GEOM_DIR = f"{REPO_ROOT}/data/processed"         
EMBED_DIR = f"{REPO_ROOT}/data/processed"        
FM_JSON_PATH = f"{REPO_ROOT}/data/processed/fm_profiling.json"

# Trỏ BATCH_RUN_DIR ra ngoài Root Repo, ngang hàng với /data và /core
current_time = datetime.now().strftime("%Y%m%d_%H%M")
BATCH_RUN_DIR = f"{REPO_ROOT}/experiments/batch_run_{current_time}"
os.makedirs(BATCH_RUN_DIR, exist_ok=True)

print(f"[*] Thư mục Root: {REPO_ROOT}")
print(f"[*] Thư mục lưu trữ thực nghiệm: {BATCH_RUN_DIR}")

CONFIG = {
    "dna_model": "evo2_1b",
    "dna_dim": 2048,
    "prot_model": "esm2_650m",
    "prot_dim": 1280,
    "batch_size": 256,
    "epochs": 15,
    "lr": 1e-4,
    "num_workers": 0
}

POOLING_STRATEGIES = ["center", "cls", "mean"]

EXPERIMENTS = [
    {"name": "PyTorch_Concat", "type": "pytorch", "fusion": "concat"},
    {"name": "PyTorch_CrossAttn", "type": "pytorch", "fusion": "cross_attention"},
    {"name": "PyTorch_Gating", "type": "pytorch", "fusion": "gating"},
    {"name": "Pure_XGBoost_Concat", "type": "xgboost_pure", "fusion": "concat"},
    {"name": "Hybrid_CrossAttn_XGBoost", "type": "hybrid", "fusion": "cross_attention"}
]

# ==============================================================================
# 2. CÁC HÀM TIỆN ÍCH LÕI (CORE HELPERS)
# ==============================================================================
def get_dataloaders(split_name, pooling_strategy, is_train=True):
    dataset = VariantFusionDataset(
        bio_parquet_path=f"{BIO_DIR}/{split_name}_normalized.parquet",
        dna_geom_path=f"{GEOM_DIR}/{split_name}/{CONFIG['dna_model']}_geom_norm.parquet",
        prot_geom_path=f"{GEOM_DIR}/{split_name}/{CONFIG['prot_model']}_geom_norm.parquet",
        dna_pt_path=f"{EMBED_DIR}/{split_name}/{CONFIG['dna_model']}_{pooling_strategy}.pt",
        prot_pt_path=f"{EMBED_DIR}/{split_name}/{CONFIG['prot_model']}_{pooling_strategy}.pt",
        is_train=True
    )
    return DataLoader(dataset, batch_size=CONFIG["batch_size"], shuffle=is_train, 
                      num_workers=CONFIG["num_workers"], pin_memory=True)

def extract_features_for_ml(model, dataloader, extract_f_global=True):
    """[BẢN VÁ 1] Trích xuất dữ liệu ra Numpy Arrays chuẩn bị cho XGBoost/Hybrid"""
    model.eval()
    all_f_global, all_v_dna, all_v_prot, all_bg, all_labels, all_vids = [], [], [], [], [], []
    
    with torch.no_grad():
        for batch in dataloader:
            v_dna, v_prot = batch["v_dna"].to(DEVICE), batch["v_prot"].to(DEVICE)
            bg = torch.cat([batch["bio_features"], batch["geom_features"]], dim=-1).to(DEVICE)
            
            if extract_f_global:
                f_global = model(v_dna, v_prot, batch["bio_features"].to(DEVICE), batch["geom_features"].to(DEVICE), return_features=True)
                all_f_global.append(f_global.cpu())
                
            all_v_dna.append(v_dna.cpu())
            all_v_prot.append(v_prot.cpu())
            all_bg.append(bg.cpu())
            all_labels.append(batch["label"].cpu())
            all_vids.extend(batch["variant_id"]) # Yêu cầu đã cấu hình ở dataset.py
            
    f_glob_out = torch.cat(all_f_global).numpy() if extract_f_global else None
    return (f_glob_out, torch.cat(all_v_dna).numpy(), torch.cat(all_v_prot).numpy(), 
            torch.cat(all_bg).numpy(), torch.cat(all_labels).squeeze(-1).numpy(), all_vids)

def save_test_probabilities(vids, y_true, y_probs, y_preds, out_path):
    df = pd.DataFrame({
        "Variant_ID": vids,
        "True_Label": y_true,
        "Predicted_Probability": y_probs,
        "Prediction_Class": y_preds
    })
    df.to_csv(out_path, index=False)
    print(f"  -> Đã lưu bảng xác suất chi tiết: {os.path.basename(out_path)}")

### The Master Loop

In [ ]:
global_leaderboard_metrics = []
global_leaderboard_profiling = []

for pooling in POOLING_STRATEGIES:
    print("\n" + "="*80)
    print(f"[>>>>] KHỞI ĐỘNG CHIẾN LƯỢC POOLING: {pooling.upper()} [<<<<]")
    print("="*80)
    
    train_loader = get_dataloaders("train", pooling, is_train=True)
    val_loader = get_dataloaders("val", pooling, is_train=False)
    test_loader = get_dataloaders("test_clinvar_hq", pooling, is_train=False)
    
    for exp in EXPERIMENTS:
        exp_name = f"{exp['name']}_{pooling}"
        exp_dir = f"{BATCH_RUN_DIR}/{exp_name}"
        
        os.makedirs(f"{exp_dir}/checkpoints", exist_ok=True)
        os.makedirs(f"{exp_dir}/tensorboard_logs", exist_ok=True)
        best_model_path = f"{exp_dir}/checkpoints/best_model.pth"
        
        print(f"\n[>>>] Đang chạy thí nghiệm: {exp_name}...")
        
        profiler = FusionEvaluatorProfiler(tensorboard_dir=f"{exp_dir}/tensorboard_logs", fm_profile_json=FM_JSON_PATH, device=DEVICE)
        
        model = MultiStrategyFusionModel(
            dna_in_dim=CONFIG["dna_dim"], prot_in_dim=CONFIG["prot_dim"], fusion_strategy=exp["fusion"]
        ).to(DEVICE)
        
        test_metrics, e2e_profiling = {}, {}
        vids_t, y_true_t, y_probs_t, y_preds_t = [], [], [], []
        
        # ---------------------------------------------------------------------
        # HƯỚNG 1: PYTORCH END-TO-END
        # ---------------------------------------------------------------------
        if exp["type"] == "pytorch":
            dummy_batch = next(iter(val_loader))
            d_in = (dummy_batch["v_dna"][:1].to(DEVICE), dummy_batch["v_prot"][:1].to(DEVICE), 
                    dummy_batch["bio_features"][:1].to(DEVICE), dummy_batch["geom_features"][:1].to(DEVICE))
            profiler.profile_pytorch_fusion(model, d_in)
            
            optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"])
            criterion = nn.BCEWithLogitsLoss()
            best_val_mcc = -1.0
            
            for epoch in range(CONFIG["epochs"]):
                model.train()
                total_loss = 0
                for batch in train_loader:
                    optimizer.zero_grad()
                    logits = model(batch["v_dna"].to(DEVICE), batch["v_prot"].to(DEVICE), 
                                   batch["bio_features"].to(DEVICE), batch["geom_features"].to(DEVICE), return_features=False)
                    loss = criterion(logits, batch["label"].to(DEVICE))
                    loss.backward()
                    optimizer.step()
                    total_loss += loss.item()
                    
                # Evaluate Val
                model.eval()
                all_probs_v, all_preds_v, all_labels_v = [], [], []
                with torch.no_grad():
                    for batch in val_loader:
                        logits = model(batch["v_dna"].to(DEVICE), batch["v_prot"].to(DEVICE), batch["bio_features"].to(DEVICE), batch["geom_features"].to(DEVICE))
                        probs = torch.sigmoid(logits)
                        all_probs_v.append(probs.cpu())
                        all_preds_v.append((probs > 0.5).float().cpu())
                        all_labels_v.append(batch["label"].cpu())
                        
                val_metrics = profiler.compute_metrics(torch.cat(all_labels_v).squeeze(-1).numpy(), torch.cat(all_probs_v).squeeze(-1).numpy(), torch.cat(all_preds_v).squeeze(-1).numpy())
                profiler.log_epoch_scalars(epoch, total_loss/len(train_loader), 0.0, val_metrics)
                
                if val_metrics["MCC"] > best_val_mcc:
                    best_val_mcc = val_metrics["MCC"]
                    torch.save(model.state_dict(), best_model_path)
            
            # --- TEST INFERENCE ---
            model.load_state_dict(torch.load(best_model_path, weights_only=True)) # [BẢN VÁ 3]
            model.eval()
            profiler.reset_memory_stats()
            
            all_probs_t, all_preds_t, all_labels_t = [], [], []
            with torch.no_grad():
                for batch in test_loader:
                    v_dna, v_prot = batch["v_dna"].to(DEVICE), batch["v_prot"].to(DEVICE)
                    bio, geom = batch["bio_features"].to(DEVICE), batch["geom_features"].to(DEVICE)
                    
                    profiler.tic_inference()
                    logits = model(v_dna, v_prot, bio, geom)
                    profiler.toc_inference()
                    
                    probs = torch.sigmoid(logits)
                    all_probs_t.append(probs.cpu())
                    all_preds_t.append((probs > 0.5).float().cpu())
                    all_labels_t.append(batch["label"].cpu())
                    vids_t.extend(batch["variant_id"])
                    
            profiler.finalize_fusion_inference_profiling(len(test_loader.dataset))
            y_probs_t, y_preds_t = torch.cat(all_probs_t).squeeze(-1).numpy(), torch.cat(all_preds_t).squeeze(-1).numpy()
            y_true_t = torch.cat(all_labels_t).squeeze(-1).numpy()
            
            test_metrics = profiler.compute_metrics(y_true_t, y_probs_t, y_preds_t)
            e2e_profiling = profiler.get_e2e_profiling(CONFIG["dna_model"], CONFIG["prot_model"])
            
        # ---------------------------------------------------------------------
        # HƯỚNG 2: XGBOOST THUẦN TÚY (PURE ML)
        # ---------------------------------------------------------------------
        elif exp["type"] == "xgboost_pure":
            profiler.fusion_metrics["gflops_per_sample"] = 0.0
            profiler.fusion_metrics["num_parameters"] = 0
            profiler.fusion_metrics["param_memory_mb"] = 0.0
            
            _, dna_tr, prot_tr, bg_tr, y_tr, _ = extract_features_for_ml(model, train_loader, False)
            _, dna_vl, prot_vl, bg_vl, y_vl, _ = extract_features_for_ml(model, val_loader, False)
            _, dna_ts, prot_ts, bg_ts, y_ts, vids_t = extract_features_for_ml(model, test_loader, False)
            
            xgb_manager = XGBoostFusionManager(pca_components=256)
            xgb_manager.train_pure(dna_tr, prot_tr, bg_tr, y_tr, dna_vl, prot_vl, bg_vl, y_vl)
            
            profiler.reset_memory_stats()
            profiler.tic_inference()
            y_probs_t, y_preds_t, _ = xgb_manager.predict_pure(dna_ts, prot_ts, bg_ts)
            profiler.toc_inference()
            
            profiler.finalize_fusion_inference_profiling(len(test_loader.dataset))
            y_true_t = y_ts
            test_metrics = profiler.compute_metrics(y_true_t, y_probs_t, y_preds_t)
            e2e_profiling = profiler.get_e2e_profiling(CONFIG["dna_model"], CONFIG["prot_model"])
            
            del xgb_manager, dna_tr, prot_tr, bg_tr, dna_vl, prot_vl, bg_vl, dna_ts, prot_ts, bg_ts

        # ---------------------------------------------------------------------
        # HƯỚNG 3: HYBRID TRÍCH XUẤT F_GLOBAL -> XGBOOST
        # ---------------------------------------------------------------------
        elif exp["type"] == "hybrid":
            profiler.profile_pytorch_fusion(model, (next(iter(val_loader))["v_dna"][:1].to(DEVICE), next(iter(val_loader))["v_prot"][:1].to(DEVICE), next(iter(val_loader))["bio_features"][:1].to(DEVICE), next(iter(val_loader))["geom_features"][:1].to(DEVICE)))
            
            optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"])
            criterion = nn.BCEWithLogitsLoss()
            
            print("  [Hybrid] Train lướt 2 Epochs để định hình Fusion...")
            model.train()
            for _ in range(2):
                for batch in train_loader:
                    optimizer.zero_grad()
                    loss = criterion(model(batch["v_dna"].to(DEVICE), batch["v_prot"].to(DEVICE), batch["bio_features"].to(DEVICE), batch["geom_features"].to(DEVICE)), batch["label"].to(DEVICE))
                    loss.backward()
                    optimizer.step()
                    
            f_glob_tr, _, _, _, y_tr, _ = extract_features_for_ml(model, train_loader, True)
            f_glob_vl, _, _, _, y_vl, _ = extract_features_for_ml(model, val_loader, True)
            f_glob_ts, _, _, _, y_ts, vids_t = extract_features_for_ml(model, test_loader, True)
            
            xgb_manager = XGBoostFusionManager()
            xgb_manager.train_hybrid(f_glob_tr, y_tr, f_glob_vl, y_vl)
            
            profiler.reset_memory_stats()
            profiler.tic_inference()
            y_probs_t, y_preds_t, _ = xgb_manager.predict_hybrid(f_glob_ts)
            profiler.toc_inference()
            
            profiler.finalize_fusion_inference_profiling(len(test_loader.dataset))
            y_true_t = y_ts
            test_metrics = profiler.compute_metrics(y_true_t, y_probs_t, y_preds_t)
            e2e_profiling = profiler.get_e2e_profiling(CONFIG["dna_model"], CONFIG["prot_model"])
            
            del xgb_manager, f_glob_tr, f_glob_vl, f_glob_ts

        # =====================================================================
        # 4. LƯU TRỮ VÀ GHI LOG (CHUNG CHO MỌI HƯỚNG)
        # =====================================================================
        with open(f"{exp_dir}/config.json", 'w') as f:
            json.dump({**CONFIG, "pooling": pooling, "experiment": exp["name"]}, f, indent=4)
            
        with open(f"{exp_dir}/metrics.json", 'w') as f:
            json.dump({"classification": test_metrics, "profiling": e2e_profiling}, f, indent=4)
            
        # [BẢN VÁ 2] Xuất file xác suất cho TẤT CẢ thí nghiệm
        save_test_probabilities(vids_t, y_true_t, y_probs_t, y_preds_t, f"{exp_dir}/test_probabilities.csv")
            
        profiler.log_hparams(hparam_dict={"fusion": exp["fusion"], "pooling": pooling, "lr": CONFIG["lr"]}, 
                             final_metrics={"MCC_Test": test_metrics.get("MCC", 0)})
        profiler.close()
        
        global_leaderboard_metrics.append({"Experiment": exp_name, **test_metrics})
        global_leaderboard_profiling.append({"Experiment": exp_name, **e2e_profiling})
        
        # Giải phóng model
        del model, profiler
        torch.cuda.empty_cache()
        gc.collect()
        
    del train_loader, val_loader, test_loader
    gc.collect()

# ==============================================================================
# 5. XUẤT BẢNG XẾP HẠNG TỔNG HỢP (THE GLOBAL LEADERBOARDS)
# ==============================================================================
print("\n" + "="*80)
print("[LEADERBOARD 1] HIỆU SUẤT PHÂN LOẠI (CLASSIFICATION METRICS)")
print("="*80)
df_metrics = pd.DataFrame(global_leaderboard_metrics).sort_values(by="MCC", ascending=False)
display(df_metrics)
df_metrics.to_csv(f"{BATCH_RUN_DIR}/global_leaderboard_metrics.csv", index=False)

print("\n" + "="*80)
print("[LEADERBOARD 2] HIỆU NĂNG TÀI NGUYÊN (END-TO-END PROFILING)")
print("="*80)
df_profiling = pd.DataFrame(global_leaderboard_profiling)
display(df_profiling)
df_profiling.to_csv(f"{BATCH_RUN_DIR}/global_leaderboard_profiling.csv", index=False)

print(f"\n[THÀNH CÔNG] Toàn bộ báo cáo đã được lưu trữ vĩnh viễn tại: {BATCH_RUN_DIR}")